# Intent Classifier — Fine-tune DistilBERT on J&J Supply Chain Data

Trains on `supply_chain_train_full.csv` (350 samples, 7 classes).  
Evaluates on `supply_chain_test.csv` (40 samples).  
Saves model to `../models/intent_classifier/` for use in the pipeline.

## Step 1 — Install dependencies

In [1]:
%pip install transformers torch scikit-learn datasets accelerate --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\Abcom\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


## Step 2 — Imports & device check

In [2]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import classification_report, accuracy_score

print('PyTorch   :', torch.__version__)
print('CUDA      :', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device    :', device)

c:\Users\Abcom\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch   : 2.11.0+cpu
CUDA      : False
Device    : cpu


## Step 3 — Load data

In [3]:
DATA_DIR  = Path(r'C:\Users\Abcom\J&J\j&j\data')
MODEL_DIR = Path(r'C:\Users\Abcom\J&J\j&j\models\intent_classifier')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / 'supply_chain_train_full.csv')
test_df  = pd.read_csv(DATA_DIR / 'supply_chain_test.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print()
print('Label distribution (train):')
print(train_df['label'].value_counts())

Train shape: (350, 2)
Test shape : (40, 2)

Label distribution (train):
label
kpi_lookup            50
trend_analysis        50
comparison            50
anomaly_detection     50
root_cause            50
forecast              50
operational_status    50
Name: count, dtype: int64


## Step 4 — Encode labels

In [4]:
labels   = sorted(train_df['label'].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

print('Classes:', labels)

train_df['label_id'] = train_df['label'].map(label2id)
test_df['label_id']  = test_df['label'].map(label2id)

# Verify no NaN label_ids (all test labels must be in train)
assert test_df['label_id'].notna().all(), 'Unknown labels in test set!'
print('All labels valid.')

Classes: ['anomaly_detection', 'comparison', 'forecast', 'kpi_lookup', 'operational_status', 'root_cause', 'trend_analysis']
All labels valid.


## Step 5 — Tokenize

In [5]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=64,
    )

train_ds = Dataset.from_pandas(
    train_df[['text', 'label_id']].rename(columns={'label_id': 'labels'})
)
test_ds = Dataset.from_pandas(
    test_df[['text', 'label_id']].rename(columns={'label_id': 'labels'})
)

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format(type='torch',  columns=['input_ids', 'attention_mask', 'labels'])

print('Train tokenized:', len(train_ds))
print('Test tokenized :', len(test_ds))

Map: 100%|██████████| 40/40 [00:00<00:00, 4098.50 examples/s]

Train tokenized: 350
Test tokenized : 40


## Step 6 — Fine-tune DistilBERT (8 epochs)

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'accuracy': accuracy_score(label_ids, preds)}

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=20,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('Training started — this takes ~15-25 min on CPU...')
trainer.train()
print('Training complete!')

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3526.02it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training started — this takes ~15-25 min on CPU...


c:\Users\Abcom\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.920840,1.807428,0.550000
2,1.670154,1.297919,0.900000
3,1.210429,0.788866,0.975000
4,0.803721,0.497329,0.975000
5,0.519893,0.336456,1.000000
6,0.372617,0.244669,1.000000
7,0.294321,0.206593,1.000000
8,0.236833,0.194920,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]
c:\Users\Abcom\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]
c:\Users\Abcom\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]
c:\Users\Abcom\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

Training complete!


## Step 7 — Evaluate on test set

In [12]:
preds_output = trainer.predict(test_ds)
pred_ids = np.argmax(preds_output.predictions, axis=1)
true_ids = np.array(test_ds['labels'])

print('=== TEST SET RESULTS ===')
print(f'Overall accuracy: {accuracy_score(true_ids, pred_ids):.2%}')
print()
print(classification_report(true_ids, pred_ids, target_names=labels))

c:\Users\Abcom\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


=== TEST SET RESULTS ===
Overall accuracy: 100.00%

                    precision    recall  f1-score   support

 anomaly_detection       1.00      1.00      1.00         5
        comparison       1.00      1.00      1.00         5
          forecast       1.00      1.00      1.00         5
        kpi_lookup       1.00      1.00      1.00        10
operational_status       1.00      1.00      1.00         5
        root_cause       1.00      1.00      1.00         5
    trend_analysis       1.00      1.00      1.00         5

          accuracy                           1.00        40
         macro avg       1.00      1.00      1.00        40
      weighted avg       1.00      1.00      1.00        40



## Step 8 — Save model

In [13]:
model.save_pretrained(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f'Model saved to: {MODEL_DIR.resolve()}')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]

Model saved to: C:\Users\Abcom\J&J\j&j\models\intent_classifier


## Step 9 — Smoke test all 7 intents

In [14]:
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline(
    'text-classification',
    model=str(MODEL_DIR),
    tokenizer=str(MODEL_DIR),
    device=-1,
)

smoke_tests = [
    ('What is the OTIF rate this week?',              'kpi_lookup'),
    ('Show OTIF trend over last 30 days',             'trend_analysis'),
    ('Compare OTIF between Mumbai and Delhi',         'comparison'),
    ('Why did load rejection rate spike this week?',  'root_cause'),
    ('Did OTIF drop suddenly this month?',            'anomaly_detection'),
    ('Predict OTIF for next month',                   'forecast'),
    ('How many shipments are currently in transit?',  'operational_status'),
]

print(f'{"Query":<52} {"Expected":<22} {"Predicted":<22} OK')
print('-' * 105)
all_ok = True
for query, expected in smoke_tests:
    result    = pipe(query)[0]
    predicted = result['label']
    score     = result['score']
    ok        = '✓' if predicted == expected else '✗'
    if predicted != expected:
        all_ok = False
    print(f'{query:<52} {expected:<22} {predicted:<22} {ok}  ({score:.2f})')

print()
if all_ok:
    print('All smoke tests passed! Model is ready.')
    print(f'Pipeline will auto-load it from: {MODEL_DIR.resolve()}')
else:
    print('Some tests failed. Consider more epochs or additional training data.')

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3410.96it/s]


Query                                                Expected               Predicted              OK
---------------------------------------------------------------------------------------------------------
What is the OTIF rate this week?                     kpi_lookup             kpi_lookup             ✓  (0.75)
Show OTIF trend over last 30 days                    trend_analysis         trend_analysis         ✓  (0.76)
Compare OTIF between Mumbai and Delhi                comparison             comparison             ✓  (0.82)
Why did load rejection rate spike this week?         root_cause             root_cause             ✓  (0.83)
Did OTIF drop suddenly this month?                   anomaly_detection      anomaly_detection      ✓  (0.82)
Predict OTIF for next month                          forecast               forecast               ✓  (0.81)
How many shipments are currently in transit?         operational_status     operational_status     ✓  (0.69)

All smoke tests passed! Mode